<a href="https://colab.research.google.com/github/galipaydin/pes517/blob/main/notebooks/lab02_prompt_to_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PES 517 · Lab 2 — Prompt-to-notebook: make an LLM write your analysis
**Artificial Intelligence Applications in Sport · Week 2**

⏱️ About 60 minutes · 🧪 This notebook creates its own data — nothing can fail to download · 🤖 You need an LLM open in another tab

> **First:** `File → Save a copy in Drive`. Cells marked ✏️ **EDIT ME** are the ones you change.


### The point of this lab

You are not going to be taught Python. You are going to be taught how to **get working Python out of a language
model and check that it is right** — which is the skill that carries you through the remaining twelve weeks.

Below is a deliberately messy file, of the kind a strength coach actually hands you. Your job is to produce five
things from it. You may not write the code yourself; you must obtain it from ChatGPT, Claude or Gemini.

**The friction is the lesson.** When the code fails — and it will — paste the *entire* error message back into the
model and say “this is the error, fix it”. Do that until it runs. Keep every attempt: the transcript is graded.


---
## Block 1 — Setup


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
RNG = np.random.default_rng(2)
print('Setup complete.')


---
## Block 2 — The file you were given

Run this cell. It writes `training_load_messy.csv` into the session and shows you the first rows.


In [ ]:
n_ath, n_days = 12, 100
ids_pretty = [f'A{i:02d}' for i in range(n_ath)]
rows = []
for i in range(n_ath):
    for d in range(n_days):
        date = pd.Timestamp('2026-09-01') + pd.Timedelta(days=d)
        # dates recorded in two different formats by two different assistants
        date_str = date.strftime('%d/%m/%Y') if i % 2 == 0 else date.strftime('%Y-%m-%d')
        # athlete ids typed inconsistently
        aid = ids_pretty[i] if RNG.random() < .7 else ids_pretty[i].lower().replace('a0','a').replace('a','a')
        mins = float(np.clip(RNG.normal(80, 18), 25, 140))
        # duration sometimes in minutes, sometimes as h:mm
        dur = f'{int(mins)//60}:{int(mins)%60:02d}' if RNG.random() < .35 else str(round(mins))
        rpe = '' if RNG.random() < .12 else str(round(float(np.clip(RNG.normal(6.1, 1.6), 1, 10)), 1))
        rows.append({'Date': date_str, 'Athlete': aid, 'Duration': dur, 'RPE': rpe,
                     'Notes': RNG.choice(['', '', '', 'late', 'modified session'])})

messy = pd.DataFrame(rows)
messy.to_csv('training_load_messy.csv', index=False)

print('Saved training_load_messy.csv —', len(messy), 'rows')
print('\nColumn types as pandas reads them:')
print(messy.dtypes)
messy.head(12)


> **Look at it before you ask for anything.** How many date formats do you see? How is `Duration` written?
> What is in `RPE` when nobody recorded it? Write those down — a model can only fix problems you describe to it.


---
## Block 3 — Your five tasks

Get an LLM to give you code that does each of these. Paste its code into the cell underneath, run it, and iterate.

| # | Task |
|---|---|
| 1 | Read `training_load_messy.csv` into a DataFrame |
| 2 | Parse `Date` correctly **even though there are two formats** |
| 3 | Standardise `Athlete` so `A03` and `a3` become the same athlete |
| 4 | Convert `Duration` to a number of minutes, handling both `85` and `1:25` |
| 5 | Compute `session_load = RPE × duration_min`, then plot weekly total load per athlete with a 4-week rolling mean |

### A prompt that works better than “clean my data”

```
You are helping a sport scientist who does not write code.
I have a CSV with these exact columns and these exact example values:
  Date: '03/09/2026' and '2026-09-03'   (two formats in the same column)
  Athlete: 'A03' and 'a3'               (same athlete, typed two ways)
  Duration: '85' and '1:25'             (minutes, or h:mm)
  RPE: '6.5' and ''                     (empty when not recorded)
Write pandas code that ... [your task here].
Add a comment on every line explaining what it does.
Do not use any library other than pandas, numpy and matplotlib.
```

**Give it your schema, not a description of your data.** That single change is most of the difference.


### ✏️ Paste the LLM's code here — Tasks 1–4 (cleaning)


In [ ]:
# ✏️ EDIT ME — paste the code you obtained, then run it.
# Expected result: a DataFrame called `clean` with columns
#   date (datetime64), athlete (string), duration_min (float), rpe (float)




### Checkpoint — run this to see whether your cleaning worked

Do **not** edit this cell. It checks your result and tells you what is still wrong.


In [ ]:
def checkpoint(clean):
    ok = True
    try:
        assert 'date' in clean and 'athlete' in clean and 'duration_min' in clean and 'rpe' in clean
    except Exception:
        print('✗ Missing one of: date, athlete, duration_min, rpe'); return
    if not np.issubdtype(clean['date'].dtype, np.datetime64):
        print('✗ `date` is not a datetime column'); ok = False
    else:
        span = (clean.date.max() - clean.date.min()).days
        print(f'✓ dates parsed — span {span} days' if 20 < span < 200
              else f'✗ date span is {span} days, which cannot be right (two formats mixed up?)')
        ok &= 20 < span < 200
    n_ath = clean['athlete'].nunique()
    print(f'{"✓" if n_ath == 12 else "✗"} {n_ath} unique athletes (should be 12)'); ok &= n_ath == 12
    d = pd.to_numeric(clean['duration_min'], errors='coerce')
    print(f'{"✓" if d.between(20,150).mean() > .95 else "✗"} durations: min {d.min():.0f}, max {d.max():.0f} minutes'); ok &= d.between(20,150).mean() > .95
    r = pd.to_numeric(clean['rpe'], errors='coerce')
    print(f'✓ RPE: {r.isna().sum()} missing values left as NaN (that is correct — do not fill them with 0)')
    print('\n' + ('ALL CHECKS PASSED — go on to the plot.' if ok else 'Not there yet. Paste the failures above back into the LLM.'))

try:
    checkpoint(clean)
except NameError:
    print('No DataFrame called `clean` yet — paste the LLM code in the cell above first.')


### ✏️ Paste the LLM's code here — Task 5 (weekly load plot)


In [ ]:
# ✏️ EDIT ME — weekly total session load per athlete, with a 4-week rolling mean




---
## Block 4 — Hallucination hunt 🕵️

Three statements about this dataset are below. **Two are true, one is false but entirely plausible.**

1. *“Athlete A00 has the highest mean RPE in the squad.”*
2. *“Roughly one RPE value in eight was never recorded.”*
3. *“Sessions longer than two hours make up about 5% of the file.”*

Ask your LLM to verify each one **from the data** — paste it the head of the cleaned frame and ask for code that
checks the claim. Then run the cell below, which checks them properly, and compare.


In [ ]:
# the honest check (do not show this to the model until after it has answered)
try:
    r = pd.to_numeric(clean['rpe'], errors='coerce')
    d = pd.to_numeric(clean['duration_min'], errors='coerce')
    top = clean.assign(r=r).groupby('athlete')['r'].mean().idxmax()
    print(f'1. Athlete with the highest mean RPE : {top}')
    print(f'2. Proportion of RPE never recorded  : {r.isna().mean():.1%}')
    print(f'3. Sessions longer than 120 minutes  : {(d > 120).mean():.1%}')
except NameError:
    print('Finish the cleaning first.')


**Record what happened.** Did the model check the claim, or agree with it? Did it produce a number without
running any code? Did it invent a plausible percentage? That behaviour, not the answer, is the finding.


---
## Interpretation ✍️ **(graded)**

About 150 words:

1. Describe **the moment the model was confidently wrong** — quote what it said.
2. How many prompts did it take to get working cleaning code? What finally made the difference?
3. If you had accepted its first answer without running it, what would have gone into your thesis?

---

**Your answer:**

*(replace this line)*


---
## AI Use Log — full transcript required for this lab

Paste **every** prompt and every error you pasted back, in order. This is the largest part of the mark for Lab 2.

| # | My prompt | What it returned | Did it run? | What I did next |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |


---
### Submitting

1. `File → Save a copy in Drive` · 2. `Share → Anyone with the link → Viewer` · 3. Paste the link into ODTÜClass.
